# **T4.5.2 Geotagging of texts** 

* This workflow is part of the [ATRIUM](https://atrium-research.eu/) project.

* This notebook supports **`.xml/tei`**, **`.txt`** and **`.pdf`** file formats.

* In this example, we will use a preprocessed subset `pausanias.xml`, `pausanias.txt` and `pausanias.pdf` file, downloaded from the [Digital Periegesis](https://www.periegesis.org/en/reports.php?projectid=1).

* In this example, we will also use [ToposText](https://topostext.org/) as our knowledege base linking. You can use a different KB based on your texts and objectives.

* In this example we will be using the [llama.cpp](https://github.com/ggml-org/llama.cpp) framework.

#### Requirements

In [ ]:
# If you plan to use ToposText as a gazetter then uncomment the 4 lines below:
#!wget https://github.com/atrium-research/T4.5.2_Geotagging_of_texts/releases/download/v.1/topostext.index -P ./data/topostext
#!wget https://github.com/atrium-research/T4.5.2_Geotagging_of_texts/releases/download/v.1/topostext_meta.pkl -P ./data/topostext
#!wget https://github.com/atrium-research/T4.5.2_Geotagging_of_texts/releases/download/v.1/ToposText_gazetteer.json -P ./data/topostext
#!wget https://raw.githubusercontent.com/atrium-research/T4.5.2_Geotagging_of_texts/refs/heads/main/notebook_examples/config/config.yaml -P ./data/config

In [ ]:
# Restart the kernel after the first installation
%pip install -r "https://raw.githubusercontent.com/atrium-research/T4.5.2_Geotagging_of_texts/refs/heads/main/notebook_examples/requirements.txt"
%pip install --no-deps https://huggingface.co/NikosKprl/en_deberta_v3_base_ner_historical_place/resolve/main/en_deberta_v3_base_ner_historical_place-1.0-py3-none-any.whl

In [ ]:
import os

# Create the folders
# Move your .txt, .xml/tei or .pdf file inside the 'data' folder
os.makedirs("outputs", exist_ok=True)
os.makedirs("data", exist_ok=True)

### **Step 1: Perform Name Entity Recognition (NER) using our pre-trained model**

In the .txt dataset,each paragraph is denoted by 2 newlines `\n\n`.
Change the next cell structure according to your own dataset.

In [ ]:
# Run this on your termimal if you plan to use PDFs
#llama-server -hf ggml-org/GLM-OCR-GGUF:F16

In [ ]:
import spacy
from tqdm import tqdm
import srsly
from lxml import etree as ET
from glmocr import GlmOcr

# Setups NER location path for pre-trained model
ner_location_model_path = "en_deberta_v3_base_ner_historical_place"

# Setup input and output paths
input_path = "data/pausanias.txt" # OR "data/pausanias.xml" OR "data/pausanias.pdf"
ner_output_path = "./outputs/ner_data.jsonl"

# Check if the input path is .txt, .xml or .pdf file
if input_path.endswith(".txt"):
    with open(input_path, "r", encoding="utf-8") as f:
        input_data = [{"text":i.replace("\n","").strip()} for i in f.readlines() if i != "\n"]
    
elif input_path.endswith(".xml"):
    tree = ET.parse(input_path)
    root = tree.getroot()
    ns = {"tei": "http://www.tei-c.org/ns/1.0"}
    paragraphs = root.findall(".//tei:p", ns)

    input_data = ["".join(p.itertext()).strip() for p in paragraphs]

elif input_path.endswith(".pdf"):
    with GlmOcr(config_path="data/config/config.yaml", log_level="ERROR") as parser:
        result = parser.parse(input_path)
        final_text = [text.get("content") for element in result.json_result for text in element if text]
else:
    print("Please specify a .txt, .xml/tei or .pdf file on the input_path variable")

In [ ]:
def NER(model_path, in_data, entity):
    ner_model = spacy.load(model_path)
    annotated_data = []
    for row in tqdm(in_data, desc=f"Performing NER for {entity}"):
        sent_nlp = ner_model(row["text"])
        ner_spans = [{"start": span.start_char, "end": span.end_char, "label": entity} for span in sent_nlp.ents]
        if "spans" in row:
            row["spans"] += ner_spans
        else:
            row["spans"] = ner_spans

        annotated_data.append(row)

    return annotated_data

In [ ]:
ner_data = NER(ner_location_model_path, input_data, "PLACE")

srsly.write_jsonl(ner_output_path, ner_data)

### **Step 2: Recontext the NER predictions using a Large Language Model (LLM)**
* In this example, we are using the [Qwen3.5-9B](https://huggingface.co/Qwen/Qwen3.5-9B).
* However, you can use a different LLM which your system can support.

In [ ]:
# Run this on your termimal
#llama-server -hf unsloth/Qwen3.5-9B-GGUF:Q4_K_M --reasoning off

In [ ]:
from openai import OpenAI
import srsly
from tqdm import tqdm

# Setup input and output paths
input_data = list(srsly.read_jsonl("outputs/ner_data.jsonl"))
llm_output_path = "./outputs/llm_data.jsonl"

# Setup LLM from HuggingFace for the text generation step
model = "unsloth/Qwen3.5-9B-GGUF:Q4_K_M"

# Setup OpenAI endpoint
client = OpenAI(
    base_url="http://localhost:8080/v1",
    api_key="1234"
)

In [ ]:
# This is the instructions the LLM will use, you can change it according to your task

system_prompt = """
You are an expert historian and archaeologist.

You will receive:
- Mention: a referenced entity (building, temple, city, region, island, country, monument, harbor, person, or object)
- Context: a short text snippet used only for disambiguation

Task:
Identify the most likely real-world entity referred to by the Mention using the Context, and produce a clean, retrieval-optimized entity description.

Important:
- Use the Context ONLY for disambiguation.
- Do NOT describe or refer to the Context or source text.
- Do NOT explain reasoning or how the entity appears in the text.
- Do NOT write narrative or historical essays.

Output format:
One single sentence describing the entity.

Style rules:
- Be factual and compact
- Include: entity type + location + key identifying feature
- Avoid extra historical narrative details unless essential for identification
- Keep it consistent across all outputs

Output examples:

Kantharos: Kantharos is the main ancient harbor of Piraeus in Athens, Greece.

Athens: Athens is a major city in Greece and the historical center of ancient Greek civilization.

Pompeion: The Pompeion is a public building in ancient Athens used for organizing religious processions.
"""

In [ ]:
def recontext(system_prompt, mention, text, client, model):
    user_prompt = f'Mention: "{mention}" Context: {text}'
    messages = [{"role": "system", "content": system_prompt},{"role": "user", "content": user_prompt}]
    
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=81920,
        temperature=1.0,
        top_p=0.95,
        presence_penalty=1.5,
        extra_body={
            "repetition-penalty":1.0,
            "top-k":20,
            "min-p":0.0
        }
    )

    return response.choices[0].message.content

In [ ]:
for row in tqdm(input_data, desc=f"Running {model} model to generate accurate descriptions"):
    for element in row["spans"]:
        text = row["text"]
        mention = text[element["start"]:element["end"]]
        llm_text = recontext(system_prompt, mention, text, client, model)
        element.update({"recontext":llm_text})
        
    srsly.write_jsonl(llm_output_path, [row], append=True, append_new_line=False)

### **Step 3: Indexing & fast approximate retrieval**

* The FAISS index was built from [ToposText](https://topostext.org/) database and can be found [here](https://github.com/atrium-research/T4.5.2_Geotagging_of_texts/releases/tag/v.1) along with the metadata file.
* In this example, we are using the [Qwen3-Embedding-4B](https://huggingface.co/Qwen/Qwen3-Embedding-4B) embedding model.
* However, you can use a different embedding model which your system can support.

In [ ]:
# Run this on your termimal
#!llama-server -hf Qwen/Qwen3-Embedding-4B-GGUF:Q8_0 --embedding --pooling last -ub 1024 -b 1024 -c 8192

In [ ]:
import srsly
import faiss
import pickle
from tqdm import tqdm
from openai import OpenAI
import numpy as np

In [ ]:
# Setup input and output paths
input_data = list(srsly.read_jsonl("outputs/llm_data.jsonl"))
llm_output_path = "./outputs/retrieval_data.jsonl"

# Setup OpenAI endpoint
client = OpenAI(
    base_url="http://localhost:8080/v1",
    api_key="1234"
)

# Setup model from HuggingFace to generate embeddings
model = "Qwen/Qwen3-Embedding-4B-GGUF:Q8_0"

# Import ToposText gazetteer
gazetteer = srsly.read_json("data/topostext/ToposText_gazetteer.json")
index = faiss.read_index("data/topostext/topostext.index")
with open("data/topostext/topostext_meta.pkl", "rb") as f:
    metadata = pickle.load(f)

In [ ]:
for row in tqdm(input_data, desc=f"Creating embeddings with {model} model and performing retrieval"):
    for mention in row.get("spans"):
        
        array = []
        query_text = f'{mention.get("name")}: {mention.get("recontext")}'
        embedding = client.embeddings.create(
            model=model,
            input=query_text,
            encoding_format="float"
            )
        
        query_vec = np.array(embedding.data[0].embedding).reshape(1, -1)

        distances, indices = index.search(query_vec, 30)

        result_ids = metadata.get("ids")[indices]

        for id, distance in zip(result_ids[0], distances[0]):
            array.append([gazetteer["features"][list(id)[0]].get("@id").split("/")[-1], str(distance)])

        mention.update({"index_results":array})
    srsly.write_jsonl(llm_output_path, [row], append=True, append_new_line=False)

### **Step 4: Run a Reranker for better results**

* In this example, we are using the [gpt-oss-20B](https://huggingface.co/openai/gpt-oss-20b) as a reranking tool.
* However, you can use a different LLM which your system can support.

In [ ]:
import srsly
from tqdm import tqdm
from openai import OpenAI
from rapidfuzz import process, fuzz

In [ ]:
# Threshold for when to run the LLM, leave it to 0.05 if you used a strong embedding model
threshold = 0.05

# Top k from the embedding model (default is '30' which provides a good balance between speed and accuracy)
top_k = 30

# Setup input and output paths
input_data = list(srsly.read_jsonl("./outputs/retrieval_data.jsonl"))
rerank_output_path = "./outputs/rerank_data.jsonl"

# Model from HF to use as a reranker
model = "unsloth/Qwen3.5-9B-GGUF:Q4_K_M"

# Setup OpenAI endpoint
client = OpenAI(
    base_url="http://localhost:8080/v1",
    api_key="1234"
)

# System prompt to use for the LLM, change according to your specific task
system_prompt = """
You are an entity matching system.

You are given:
1. A user query
2. A list of candidate entities

Each candidate is a possible interpretation of the query.

Your task is to select the single candidate that is the best semantic match to the query.

Rules:
- Use only the information provided in the query and candidate strings.
- Do NOT guess.
- Treat each candidate as a single label with meaning.
- Choose the candidate that best matches the overall intent of the query.
- If multiple candidates seem similar, choose the one most specifically aligned with the query context.
- If none are a good match, still choose the closest one.

Output format:
- Return ONLY the exact candidate string.
- Do not add explanation, punctuation, or extra text.
"""

In [ ]:
def llm_reranker(system_prompt, documents, ids, query, model):
    messages = [{"role": "system", "content": system_prompt},{"role": "user", "content": f"Query:{query}\nDocuments:{documents}"}]
    
    response = client.chat.completions.create(
        model=model,
        messages=messages, 
        max_tokens=81920, 
        temperature=1.0,
        top_p=0.95,
        presence_penalty=1.5,
        extra_body={
            "repetition-penalty":1.0,
            "top-k":20,
            "min-p":0.0
        }
    )

    # In case the output format is not the exact candidate string, run a fuzzy match
    choice = process.extractOne(response.choices[0].message.content, documents, scorer=fuzz.WRatio)[0]
    
    index = documents.index(choice)
    return ids[index]

In [ ]:
for i in tqdm(input_data, desc=f"Running {model} LLM as a reranker"):
    for mention in i["spans"]:
        topos_text_score = [i[1] for i in mention.get("results")]
        if float(topos_text_score[0]) - float(topos_text_score[1]) <= threshold:
            topos_text_ids = [i[0] for i in mention.get("results")][:top_k]

            hash_map = [{element:desc[meta.index(f"https://topostext.org/place/{element}")]} for element in topos_text_ids]
                        
            query = mention.get('recontext')
            documents = [list(element.values())[0] for element in hash_map]
            ids = [list(element.keys())[0] for element in hash_map]

            result = llm_reranker(system_prompt, documents, ids, query, model)

            mention.update({"reranker_results":result})
        else:
            mention.update({"reranker_results":mention.get("results")[0][0]})

    srsly.write_jsonl(rerank_output_path, [i], append=True, append_new_line=False)

### **Step 5: Create the input for the annotation enviroment**
* In this example, we choose as our annotation enviroment the [Recogito Studio](https://recogitostudio.org/).
* For the standarized input format, we choose `xml/tei`.
* However, you can also choose a different input format based on your goals and annotation enviroment.

In [ ]:
import srsly
from lxml import etree
import os
from tqdm import tqdm

In [ ]:
# Make folder for the XML/TEI file(s)
os.makedirs("outputs/annotation_env")

# Setup input and output paths
input_data = list(srsly.read_jsonl("./outputs/reranker_data.jsonl"))

NS_TEI = "http://www.tei-c.org/ns/1.0"
NS_XML = "http://www.w3.org/XML/1998/namespace"
NSMAP = {None: NS_TEI}

max_per_file = 50 # Adjust this depending on how big you want each XML
uid_counter = 0

In [ ]:
chunks = [input_data[i:i + max_per_file] for i in range(0, len(input_data), max_per_file)]

for index, data_chunk in tqdm(enumerate(chunks, 1), desc="Converting files"):
    tei = etree.Element("TEI", nsmap=NSMAP, version="3.3.0")
    standoff = etree.SubElement(tei, "standOff", type="recogito_studio_annotations")
    listannotation = etree.SubElement(standoff, "listAnnotation")
    text = etree.SubElement(tei, "text")
    body = etree.SubElement(text, "body")

    counter = 1

    for i in data_chunk:
        p = etree.SubElement(body, "p")
        p.text = i.get("text")

        for mention in i.get("spans"):
            annotation = etree.SubElement(
                listannotation,
                "annotation",
                target=f"/TEI[1]/text[1]/body[1]/p[{counter}]::{mention.get('start')} "
                       f"/TEI[1]/text[1]/body[1]/p[{counter}]::{mention.get('end')}"
            )
            annotation.set(f"{{{NS_XML}}}id", f"UID-FAKE-{uid_counter}")

            topos_id = mention.get("reranker_results")

            etree.SubElement(
                annotation,
                "rs",
                ana=f"https://topostext.org/place/{topos_id}"
            )

            uid_counter += 1

        counter += 1

    tree = etree.ElementTree(tei)
    tree.write(
        f"outputs/annotation_env/annotation_output_{index}.xml",
        xml_declaration=True,
        encoding="utf-8",
        pretty_print=True
    )